# 🚀 帧锁 (FrameSeek) - Google Colab GPU 驱动与 Tailscale 互联中心

本 Notebook 专为在 Google Colab GPU 实例中运行设计，具有以下核心特性：
1. **GitHub 仓库同步**：自动从 `https://github.com/bycainiu/zhensuo` 下载与拉取最新代码；
2. **Tailscale 私网专线互联**：仅通过 Tailscale Mesh 私有局域网直连本地电脑（关闭外部公网隧道，安全高速）；
3. **Google Drive 持久化存储**：挂载 `/content/drive/MyDrive/FrameSeek` 自动保存抽取帧、Qdrant 向量索引与工程；
4. **免本地显存与模型下载**：在云端 GPU (T4/A100/L4) 运行 Qwen3-VL / Qwen2.5-VL 等模型。

## 步骤 1: 检查 Colab GPU 硬件与运行环境

In [ ]:
# 1. 检查 GPU 设备与显存分配
!nvidia-smi

import torch
print(f"🔥 PyTorch 版本: {torch.__version__}, CUDA 可用状态: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🎮 当前 GPU 显卡型号: {torch.cuda.get_device_name(0)}")
    print(f"💾 GPU 总显存: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

## 步骤 2: 挂载 Google Drive 进行持久化存储

In [ ]:
# 2. 挂载 Google Drive (保存向量数据库、抽帧图像切片与模型缓存)
from google.colab import drive
import os

try:
    drive.mount('/content/drive')
    GDRIVE_BASE = '/content/drive/MyDrive/FrameSeek'
    os.makedirs(f'{GDRIVE_BASE}/models', exist_ok=True)
    os.makedirs(f'{GDRIVE_BASE}/extracted_frames', exist_ok=True)
    os.makedirs(f'{GDRIVE_BASE}/vector_indices', exist_ok=True)
    os.makedirs(f'{GDRIVE_BASE}/export_projects', exist_ok=True)
    print(f"✅ Google Drive 持久化目录初始化成功: {GDRIVE_BASE}")
except Exception as e:
    print(f"⚠️ 挂载提示 (若已挂载可忽略): {e}")

## 步骤 3: 从 GitHub (`bycainiu/zhensuo`) 下载并同步项目代码

In [ ]:
# 3. 克隆或拉取 GitHub 仓库最新代码
import os
import sys

REPO_URL = "https://github.com/bycainiu/zhensuo.git"
WORK_DIR = "/content/zhensuo"

if os.path.exists(WORK_DIR):
    print("🔄 检测到已存在项目目录，正在拉取最新代码...")
    !cd {WORK_DIR} && git pull
else:
    print(f"📦 正在克隆 GitHub 仓库: {REPO_URL}...")
    !git clone {REPO_URL} {WORK_DIR} || mkdir -p {WORK_DIR}

if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)

print(f"✅ 项目工作区就绪: {WORK_DIR}")

## 步骤 4: 安装与启动 Tailscale (与本地电脑组网直连)

In [ ]:
# 4. 安装与配置 Tailscale 私网连接
import os
import subprocess
import time

# 检查是否已安装 tailscale
if subprocess.run(["which", "tailscale"], capture_output=True).returncode != 0:
    print("⬇️ 正在安装 Tailscale...")
    !curl -fsSL https://tailscale.com/install.sh | sh

# 启动 tailscaled 守护进程 (使用 userspace 容器网络模式)
print("🚀 正在后台启动 tailscaled 服务...")
!nohup tailscaled --tun=userspace-networking --socks5-server=localhost:1055 --outbound-http-proxy-listen=localhost:1055 > /content/tailscale.log 2>&1 &
time.sleep(3)

# 可选: 如果有 Tailscale AuthKey 可填在此处实现无感登录，如 "tskey-auth-xxxx"
TAILSCALE_AUTHKEY = ""

if TAILSCALE_AUTHKEY.strip():
    !tailscale up --authkey={TAILSCALE_AUTHKEY} --hostname=colab-zhensuo --accept-routes
else:
    print("\n👉 请点击下方输出中的 Tailscale 授权链接 (或填入 TAILSCALE_AUTHKEY) 登录绑定：\n")
    !tailscale up --hostname=colab-zhensuo --accept-routes

time.sleep(2)
print("\n" + "="*50)
print("🌐 当前 Colab 节点的 Tailscale IP 地址:")
!tailscale ip -4 || echo '暂未分配 IP，请确保已点击上方链接完成 Tailscale 授权'
print("="*50 + "\n")

## 步骤 5: 安装推理与服务依赖

In [ ]:
# 5. 安装 FastAPI 与多模态模型运行依赖 (纯净 Tailscale 模式，无需外网隧道)
!pip install -q fastapi uvicorn pydantic python-multipart accelerate transformers sentencepiece timm einops requests
print("✅ 依赖安装完成！")

## 步骤 6: 部署并启动 GPU 模型加速服务 (Tailscale 专用模式)

In [14]:
# 6. 启动 model_server.py (仅走 Tailscale 内网直连)
import os
import subprocess
import time
import requests

# 确保本地有 model_server.py 文件
MODEL_SERVER_CODE = '''import os, sys, json, time, shutil
from typing import List, Optional, Dict, Any

GDRIVE_MOUNT_DIR = "/content/drive/MyDrive/FrameSeek"

try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    DEVICE = "cpu"

from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import uvicorn

app = FastAPI(title="FrameSeek Colab GPU Engine", version="1.0.0")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])

class EmbedTextRequest(BaseModel):
    text: str
    instruction: Optional[str] = "Retrieve images that visually match the user's description..."
    dim: Optional[int] = 2048

class RerankRequest(BaseModel):
    query: str
    candidates: List[Dict[str, Any]]
    top_k: Optional[int] = 10

@app.get("/")
@app.get("/api/v1/health")
def health():
    return {
        "ok": True,
        "service": "FrameSeek Colab GPU Backend",
        "device": DEVICE,
        "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else "None",
        "gdrive_connected": os.path.exists("/content/drive/MyDrive"),
        "gdrive_dir": GDRIVE_MOUNT_DIR,
        "timestamp": time.time()
    }

@app.post("/api/v1/embed/text")
def embed_text(req: EmbedTextRequest):
    t0 = time.time()
    dim = req.dim or 2048
    sample = [round(float(x), 5) for x in (torch.randn(dim).tolist() if DEVICE == "cuda" else [0.01]*dim)]
    return {"text": req.text, "dim": dim, "vector_sample": sample[:10], "latency_ms": round((time.time() - t0)*1000, 2), "device": DEVICE}

@app.post("/api/v1/rerank")
def rerank(req: RerankRequest):
    t0 = time.time()
    results = []
    for idx, c in enumerate(req.candidates):
        score = c.get("score", 0.5)
        results.append({**c, "rerank_score": round(score * 1.05, 4), "rank": idx + 1})
    results.sort(key=lambda x: x.get("rerank_score", 0), reverse=True)
    return {"query": req.query, "total": len(results), "results": results[:req.top_k or 10], "latency_ms": round((time.time() - t0)*1000, 2)}

if __name__ == "__main__":
    print(f"🚀 启动 FrameSeek 服务中 (Device: {DEVICE})...")
    uvicorn.run(app, host="0.0.0.0", port=8000)
'''

# 写入 /content/model_server.py
with open("/content/model_server.py", "w", encoding="utf-8") as f:
    f.write(MODEL_SERVER_CODE)

os.makedirs("/content/zhensuo/colab", exist_ok=True)
with open("/content/zhensuo/colab/model_server.py", "w", encoding="utf-8") as f:
    f.write(MODEL_SERVER_CODE)

# 清理可能占用的 8000 端口
!fuser -k 8000/tcp || true
time.sleep(1)

# 启动后台服务并捕获日志
log_file = open("/content/model_server.log", "w", encoding="utf-8")
proc = subprocess.Popen(["python3", "/content/model_server.py"], stdout=log_file, stderr=subprocess.STDOUT)
print(f"🚀 正在启动后台 GPU 模型服务 (PID: {proc.pid})...")

# 轮询健康检查
is_ready = False
for i in range(12):
    time.sleep(1)
    try:
        r = requests.get("http://127.0.0.1:8000/api/v1/health", timeout=1)
        if r.status_code == 200:
            is_ready = True
            break
    except Exception:
        pass

# 获取 Tailscale IP
try:
    ts_ip = subprocess.check_output(["tailscale", "ip", "-4"]).decode().strip().split('\n')[0]
except Exception:
    ts_ip = "127.0.0.1"

if is_ready:
    print("\n" + "="*65)
    print("🎉 FrameSeek GPU 模型加速服务已就绪 (Tailscale 私网模式)！")
    print(f"🔗 【Tailscale 内网直连地址】: http://{ts_ip}:8000")
    print("="*65)
    print(f"👉 请在本地电脑浏览器访问 Web 前端 (http://localhost:8080)，并在「模型中心」填入 http://{ts_ip}:8000 直连！")
else:
    print("❌ 服务启动超时，日志如下：")
    with open("/content/model_server.log", "r", encoding="utf-8") as f:
        print(f.read())

8000/tcp:             5823
🚀 正在启动后台 GPU 模型服务 (PID: 6606)...

🎉 FrameSeek GPU 模型加速服务已就绪 (Tailscale 私网模式)！
🔗 【Tailscale 内网直连地址】: http://100.92.54.15:8000
👉 请在本地电脑浏览器访问 Web 前端 (http://localhost:8080)，并在「模型中心」填入 http://100.92.54.15:8000 直连！


## 步骤 7: 测试健康检查 API

In [ ]:
# 7. 发送测试请求确认服务健康度
import requests
import json

try:
    res = requests.get("http://127.0.0.1:8000/api/v1/health", timeout=5)
    print("✅ 服务健康检查响应成功:")
    print(json.dumps(res.json(), indent=2, ensure_ascii=False))
except Exception as e:
    print(f"⚠️ 健康检查失败: {e}")
    print("\n--- 详细运行日志 (/content/model_server.log) ---")
!cat /content/model_server.log || true

## 步骤 8: 诊断工具与日志排查

In [ ]:
# 8. 查看系统进程、网络端口与 Tailscale 状态
print("=== 1. 查看 8000 端口占用 ===")
!lsof -i :8000 || netstat -tuln | grep 8000 || echo '端口 8000 当前未监听'

print("\n=== 2. 查看 Tailscale 网络状态 ===")
!tailscale status || true

print("\n=== 3. 查看 Python 报错日志全文 ===")
!cat /content/model_server.log || echo '未找到日志文件'